In [6]:
import os
import random
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import numpy as np

seed = 7
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
random.seed(seed)
np.random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)

In [ ]:
#! pip install IProgress 

In [7]:
categories = set()

class PeopleDaily(Dataset):
    def __init__(self, data_file):
        self.data = self.load_data(data_file)
    
    def load_data(self, data_file):
        Data = {}
        with open(data_file, 'rt', encoding='utf-8') as f:
            for idx, line in enumerate(f.read().split('\n\n')):
                if not line:
                    break
                sentence, labels = '', []
                for i, item in enumerate(line.split('\n')):
                    char, tag = item.split(' ')
                    sentence += char
                    if tag.startswith('B'):
                        labels.append([i, i, char, tag[2:]]) # Remove the B- or I-
                        categories.add(tag[2:])
                    elif tag.startswith('I'):
                        labels[-1][1] = i
                        labels[-1][2] += char
                Data[idx] = {
                    'sentence': sentence, 
                    'labels': labels
                }
        return Data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

In [15]:
train_data = PeopleDaily('china-people-daily-ner-corpus/example.train')
valid_data = PeopleDaily('china-people-daily-ner-corpus/example.dev')
test_data = PeopleDaily('china-people-daily-ner-corpus/example.test')
train_data.data

{0: {'sentence': '海钓比赛地点在厦门与金门之间的海域。',
  'labels': [[7, 8, '厦门', 'LOC'], [10, 11, '金门', 'LOC']]},
 1: {'sentence': '这座依山傍水的博物馆由国内一流的设计师主持设计，整个建筑群精美而恢宏。', 'labels': []},
 2: {'sentence': '但作为一个共产党员、人民公仆，应当胸怀宽阔，真正做到“先天下之忧而忧，后天下之乐而乐”，淡化个人的名利得失和宠辱悲喜，把改革大业摆在首位，这样才能超越自我，摆脱世俗，有所作为。',
  'labels': []},
 3: {'sentence': '在发达国家，急救保险十分普及，已成为社会保障体系的重要组成部分。', 'labels': []},
 4: {'sentence': '日俄两国国内政局都充满变数，尽管日俄关系目前是历史最佳时期，但其脆弱性不言自明。',
  'labels': [[0, 0, '日', 'LOC'],
   [1, 1, '俄', 'LOC'],
   [16, 16, '日', 'LOC'],
   [17, 17, '俄', 'LOC']]},
 5: {'sentence': '克马尔的女儿让娜今年读五年级，她所在的班上有30多名同学，该班的“家委会”由10名家长组成。',
  'labels': [[0, 2, '克马尔', 'PER'],
   [6, 7, '让娜', 'PER'],
   [33, 35, '家委会', 'ORG']]},
 6: {'sentence': '参加步行的有男有女，有年轻人，也有中年人。', 'labels': []},
 7: {'sentence': '沙特队教练佩雷拉：两支队都想胜，因此都作出了最大的努力。',
  'labels': [[0, 2, '沙特队', 'ORG'], [5, 7, '佩雷拉', 'PER']]},
 8: {'sentence': '这种混乱局面导致有些海域使用者的合法权益难以得到维护。', 'labels': []},
 9: {'sentence': '鲁宾明确指出，对政府的这种指控完全没有事实根据，美国政府不想也没有向中国转让敏感技术，事实真相总有一天会大白于天下；众议院的这种做法令

In [9]:
id2label = {0:'O'}
for c in list(sorted(categories)):
    id2label[len(id2label)] = f"B-{c}"
    id2label[len(id2label)] = f"I-{c}"
label2id = {v: k for k, v in id2label.items()}

print(id2label)
print(label2id)

{0: 'O', 1: 'B-LOC', 2: 'I-LOC', 3: 'B-ORG', 4: 'I-ORG', 5: 'B-PER', 6: 'I-PER'}
{'O': 0, 'B-LOC': 1, 'I-LOC': 2, 'B-ORG': 3, 'I-ORG': 4, 'B-PER': 5, 'I-PER': 6}


In [32]:
checkpoint = "model/bert-base-chinese"
model = AutoModel.from_pretrained(checkpoint)
# tokenizer = AutoTokenizer.from_pretrained("model/bert-base-chinese-tokenizer")
# model.save_pretrained("model/bert-base-chinese")

In [12]:
tokenizer = AutoTokenizer.from_pretrained("model/bert-base-chinese")

sentence = '海钓比赛地点在厦门与金门之间的海域。'
labels = [[7, 8, '厦门', 'LOC'], [10, 11, '金门', 'LOC']]

encoding = tokenizer(sentence, truncation=True)
tokens = encoding.tokens()
label = np.zeros(len(tokens), dtype=int)
for char_start, char_end, word, tag in labels:
    token_start = encoding.char_to_token(char_start)
    token_end = encoding.char_to_token(char_end)
    label[token_start] = label2id[f"B-{tag}"]
    label[token_start+1:token_end+1] = label2id[f"I-{tag}"]

print(tokens)
print(label)
print([id2label[id] for id in label])

['[CLS]', '海', '钓', '比', '赛', '地', '点', '在', '厦', '门', '与', '金', '门', '之', '间', '的', '海', '域', '。', '[SEP]']
[0 0 0 0 0 0 0 0 1 2 0 1 2 0 0 0 0 0 0 0]
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-LOC', 'I-LOC', 'O', 'B-LOC', 'I-LOC', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


In [ ]:
def collote_fn(batch_samples):
    batch_sentence, batch_tags  = [], []
    print(batch_samples)
    for sample in batch_samples:
        batch_sentence.append(sample['sentence'])
        batch_tags.append(sample['labels'])
    batch_inputs = tokenizer(
        batch_sentence, 
        padding=True, 
        truncation=True, 
        return_tensors="pt"
    )
    batch_label = np.zeros(batch_inputs['input_ids'].shape, dtype=int)
    for s_idx, sentence in enumerate(batch_sentence):
        encoding = tokenizer(sentence, truncation=True)
        batch_label[s_idx][0] = -100
        batch_label[s_idx][len(encoding.tokens())-1:] = -100
        for char_start, char_end, _, tag in batch_tags[s_idx]:
            token_start = encoding.char_to_token(char_start)
            token_end = encoding.char_to_token(char_end)
            batch_label[s_idx][token_start] = label2id[f"B-{tag}"]
            batch_label[s_idx][token_start+1:token_end+1] = label2id[f"I-{tag}"]
    return batch_inputs, torch.tensor(batch_label)

train_dataloader = DataLoader(train_data, batch_size=4, shuffle=True, collate_fn=collote_fn)
valid_dataloader = DataLoader(valid_data, batch_size=4, shuffle=False, collate_fn=collote_fn)
test_dataloader = DataLoader(test_data, batch_size=4, shuffle=False, collate_fn=collote_fn)

In [27]:
next(iter(train_dataloader))

[{'sentence': '通过整风学习，学员普遍学到了科学真理，接受了思想洗礼，加深了对毛泽东思想这一中国的马克思主义的认识，极大地提高了马克思主义的思想政治理论水平。', 'labels': [[31, 33, '毛泽东', 'PER'], [38, 39, '中国', 'LOC'], [41, 43, '马克思', 'PER'], [56, 58, '马克思', 'PER']]}, {'sentence': '目前，全国各行各业都活跃着大批留学回国人员，广大在外留学人员也正在以多种适当的方式为国家作贡献。', 'labels': []}, {'sentence': '非洲国家每年都要接受大量的国际援助进行经济建设，是当今世界上最主要的承包劳务市场。', 'labels': [[0, 1, '非洲', 'LOC']]}, {'sentence': '由于群众的管理水平受劳力等因素制约，高密度种植造成棉田中、后期叶面指数过大，致使棉花营养生长过旺，影响棉花产量。', 'labels': []}]
{'input_ids': tensor([[ 101, 6858, 6814, 3146, 7599, 2110,  739, 8024, 2110, 1447, 3249, 6881,
         2110, 1168,  749, 4906, 2110, 4696, 4415, 8024, 2970, 1358,  749, 2590,
         2682, 3819, 4851, 8024, 1217, 3918,  749, 2190, 3688, 3813,  691, 2590,
         2682, 6821,  671,  704, 1744, 4638, 7716, 1046, 2590,  712,  721, 4638,
         6371, 6399, 8024, 3353, 1920, 1765, 2990, 7770,  749, 7716, 1046, 2590,
          712,  721, 4638, 2590, 2682, 3124, 3780, 4415, 6389, 3717, 2398,  511,
          102],
        [ 101, 4680, 1184, 

({'input_ids': tensor([[ 101, 6858, 6814, 3146, 7599, 2110,  739, 8024, 2110, 1447, 3249, 6881,
          2110, 1168,  749, 4906, 2110, 4696, 4415, 8024, 2970, 1358,  749, 2590,
          2682, 3819, 4851, 8024, 1217, 3918,  749, 2190, 3688, 3813,  691, 2590,
          2682, 6821,  671,  704, 1744, 4638, 7716, 1046, 2590,  712,  721, 4638,
          6371, 6399, 8024, 3353, 1920, 1765, 2990, 7770,  749, 7716, 1046, 2590,
           712,  721, 4638, 2590, 2682, 3124, 3780, 4415, 6389, 3717, 2398,  511,
           102],
         [ 101, 4680, 1184, 8024, 1059, 1744, 1392, 6121, 1392,  689, 6963, 3833,
          6645, 4708, 1920, 2821, 4522, 2110, 1726, 1744,  782, 1447, 8024, 2408,
          1920, 1762, 1912, 4522, 2110,  782, 1447,  738, 3633, 1762,  809, 1914,
          4905, 6844, 2496, 4638, 3175, 2466,  711, 1744, 2157,  868, 6567, 4346,
           511,  102,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
             0,    0,    0,    0,    0,    0,    0,    0,    0,    

In [30]:
batch_X, batch_y = next(iter(train_dataloader))
batch_X = batch_X.to("cuda:0")
batch_y = batch_y.to("cuda:0")
print('batch_X shape:', {k: v.shape for k, v in batch_X.items()})
print('batch_y shape:', batch_y.shape)
print(batch_X)
print(batch_y)

[{'sentence': '关于太阳系与银河系中心的距离有多种估算，被认为比较“精确”的说法是大约2．8万光年。', 'labels': [[2, 4, '太阳系', 'LOC'], [6, 8, '银河系', 'LOC']]}, {'sentence': '同时，他还呼吁全体印尼人民为了国家的发展应做出一些必要的牺牲。', 'labels': [[9, 10, '印尼', 'LOC']]}, {'sentence': '三大工程均为技术密集型企业，急需科技人才尤其是高级人才。', 'labels': []}, {'sentence': '同一天，在纽约、芝加哥、洛杉矶等许多城市，美国联邦储备银行的负责人也举行了记者招待会。', 'labels': [[5, 6, '纽约', 'LOC'], [8, 10, '芝加哥', 'LOC'], [12, 14, '洛杉矶', 'LOC'], [21, 28, '美国联邦储备银行', 'ORG']]}]
{'input_ids': tensor([[ 101, 1068,  754, 1922, 7345, 5143,  680, 7213, 3777, 5143,  704, 2552,
         4638, 6655, 4895, 3300, 1914, 4905,  844, 5050, 8024, 6158, 6371,  711,
         3683, 6772,  100, 5125, 4802,  100, 4638, 6432, 3791, 3221, 1920, 5276,
          123, 8026,  129,  674, 1045, 2399,  511,  102,    0],
        [ 101, 1398, 3198, 8024,  800, 6820, 1461, 1390, 1059,  860, 1313, 2225,
          782, 3696,  711,  749, 1744, 2157, 4638, 1355, 2245, 2418,  976, 1139,
          671,  763, 2553, 6206, 4638, 4295, 4291,  511,  102,    0,    0,    0,
      

In [33]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    checkpoint,
    id2label=id2label,
    label2id=label2id,
)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at model/bert-base-chinese and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [34]:
from torch import nn
from transformers import AutoConfig
from transformers import BertPreTrainedModel, BertModel

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f'Using {device} device')

class BertForNER(BertPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.bert = BertModel(config, add_pooling_layer=False)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(768, len(id2label))
        self.post_init()

    def forward(self, x):
        bert_output = self.bert(**x)
        sequence_output = bert_output.last_hidden_state
        sequence_output = self.dropout(sequence_output)
        logits = self.classifier(sequence_output)
        return logits

config = AutoConfig.from_pretrained(checkpoint)
model = BertForNER.from_pretrained(checkpoint, config=config).to(device)
print(model)

Some weights of BertForNER were not initialized from the model checkpoint at model/bert-base-chinese and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using cuda:0 device
BertForNER(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(21128, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1

In [35]:
outputs = model(batch_X)
outputs

tensor([[[ 0.0751,  0.3687,  0.0480,  ...,  0.3103,  0.3603,  1.0866],
         [-0.0250,  0.4512, -0.8327,  ...,  0.6660,  0.3787,  0.8671],
         [-0.3511,  1.4069,  0.2806,  ..., -0.2226,  0.3302,  0.7638],
         ...,
         [ 0.1793,  1.0676,  0.6568,  ...,  0.1529,  0.7040,  0.5769],
         [ 0.2730,  0.6959, -0.6828,  ...,  0.7127,  0.5311,  0.4817],
         [ 0.4362,  0.5337,  0.0411,  ..., -0.2287,  0.2370,  0.6100]],

        [[ 0.2362, -0.2722, -0.1088,  ..., -0.4332,  0.6258,  1.2941],
         [ 0.3430, -0.4424, -0.4146,  ...,  0.0294,  0.3738,  0.4117],
         [ 0.5730,  0.1609,  0.7185,  ..., -1.0149,  0.2577,  0.0274],
         ...,
         [ 0.7243,  0.2405,  0.2333,  ..., -0.3710,  0.1662,  0.3446],
         [ 0.5511,  0.2094,  0.3759,  ..., -0.4089,  0.1590,  0.4539],
         [ 0.9032,  0.2960,  0.2551,  ..., -0.4099,  0.0592,  0.4595]],

        [[ 0.4426,  0.0735, -0.2809,  ..., -0.4397,  0.0718,  0.8559],
         [-0.2181,  0.1135, -0.3970,  ..., -0

In [46]:
outputs.shape

torch.Size([4, 93, 7])

In [47]:
len(batch_X)

3

In [36]:
from tqdm.auto import tqdm

def train_loop(dataloader, model, loss_fn, optimizer, lr_scheduler, epoch, total_loss):
    progress_bar = tqdm(range(len(dataloader)))
    progress_bar.set_description(f'loss: {0:>7f}')
    finish_batch_num = (epoch-1) * len(dataloader)
    
    model.train()
    for batch, (X, y) in enumerate(dataloader, start=1):
        # print('start')
        # print(X)
        # print('--------')
        # print(y)
        # print('--------')
        # print(X.keys())
        X, y = X.to(device), y.to(device)
        pred = model(X)
        loss = loss_fn(pred.permute(0, 2, 1), y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()

        total_loss += loss.item()
        progress_bar.set_description(f'loss: {total_loss/(finish_batch_num + batch):>7f}')
        progress_bar.update(1)
    return total_loss

In [37]:
from seqeval.metrics import classification_report
from seqeval.scheme import IOB2

y_true = [['O', 'O', 'O', 'B-LOC', 'I-LOC', 'I-LOC', 'B-LOC', 'O'], ['B-PER', 'I-PER', 'O']]
y_pred = [['O', 'O', 'B-LOC', 'I-LOC', 'I-LOC', 'I-LOC', 'B-LOC', 'O'], ['B-PER', 'I-PER', 'O']]

print(classification_report(y_true, y_pred, mode='strict', scheme=IOB2))

              precision    recall  f1-score   support

         LOC       0.50      0.50      0.50         2
         PER       1.00      1.00      1.00         1

   micro avg       0.67      0.67      0.67         3
   macro avg       0.75      0.75      0.75         3
weighted avg       0.67      0.67      0.67         3



In [64]:
from seqeval.metrics import classification_report
from seqeval.scheme import IOB2

def test_loop(dataloader, model):
    true_labels, true_predictions = [], []

    model.eval()
    with torch.no_grad():
        for X, y in tqdm(dataloader):
            X, y = X.to(device), y.to(device)
            pred = model(X)
            predictions = pred.argmax(dim=-1).cpu().numpy().tolist()
            labels = y.cpu().numpy().tolist()
            true_labels += [[id2label[int(l)] for l in label if l != -100] for label in labels]
            true_predictions += [
                [id2label[int(p)] for (p, l) in zip(prediction, label) if l != -100]
                for prediction, label in zip(predictions, labels)
            ]
    print(classification_report(true_labels, true_predictions, mode='strict', scheme=IOB2))

In [85]:
from transformers import get_scheduler
from torch.optim import AdamW

learning_rate = 1e-5
epoch_num = 3

loss_fn = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=learning_rate)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=epoch_num*len(train_dataloader),
)

total_loss = 0.
for t in range(epoch_num):
    print(f"Epoch {t+1}/{epoch_num}\n-------------------------------")
    total_loss = train_loop(train_dataloader, model, loss_fn, optimizer, lr_scheduler, t+1, total_loss)
    test_loop(valid_dataloader, model)
print("Done!")

Epoch 1/3
-------------------------------


              precision    recall  f1-score   support

         LOC       0.96      0.96      0.96      1951
         ORG       0.91      0.92      0.91       984
         PER       0.99      0.98      0.98       884

   micro avg       0.96      0.95      0.95      3819
   macro avg       0.95      0.95      0.95      3819
weighted avg       0.96      0.95      0.95      3819

Epoch 2/3
-------------------------------


              precision    recall  f1-score   support

         LOC       0.97      0.96      0.97      1951
         ORG       0.91      0.93      0.92       984
         PER       0.99      0.98      0.98       884

   micro avg       0.96      0.96      0.96      3819
   macro avg       0.96      0.96      0.96      3819
weighted avg       0.96      0.96      0.96      3819

Epoch 3/3
-------------------------------


100%|██████████| 580/580 [00:03<00:00, 155.19it/s]


              precision    recall  f1-score   support

         LOC       0.98      0.97      0.97      1951
         ORG       0.93      0.93      0.93       984
         PER       0.99      0.98      0.99       884

   micro avg       0.97      0.96      0.96      3819
   macro avg       0.97      0.96      0.96      3819
weighted avg       0.97      0.96      0.96      3819

Done!
